In [0]:
#build
from pyspark.sql import functions as F

C = "capstone_supply_chain"
img = spark.table(f"{C}.bronze.raw_inspection_images")

silver_img = (img
    .withColumn("capture_ts", F.expr("try_cast(capture_timestamp AS TIMESTAMP)"))
    .withColumn("width_px", F.expr("try_cast(split(resolution, 'x')[0] AS INT)"))
    .withColumn("height_px", F.expr("try_cast(split(resolution, 'x')[1] AS INT)"))
    .withColumn("megapixels", F.round(F.col("width_px") * F.col("height_px") / 1000000, 2))
    .withColumn("confidence", F.expr("try_cast(confidence_score AS DOUBLE)"))
    .withColumn("is_defect", F.lower(F.trim(F.col("defect_class"))) != "none")
    # Thresholds are an assumption for this project, not a standard
    .withColumn("confidence_band",
        F.when(F.col("confidence") >= 0.95, "High")
         .when(F.col("confidence") >= 0.85, "Medium")
         .otherwise("Low"))
    .select(
        "image_id", "inspection_id", "part_number", "image_type", "camera_id",
        F.upper(F.col("format")).alias("image_format"),
        "width_px", "height_px", "megapixels",
        F.expr("try_cast(file_size_kb AS INT)").alias("file_size_kb"),
        "capture_ts", "defect_class", "is_defect",
        "confidence", "confidence_band", "description", "file_path",
        F.col("_ingested_at").alias("bronze_ingested_at"),
        F.current_timestamp().alias("silver_processed_at"),
    )
    .filter(F.col("image_id").isNotNull())
    .dropDuplicates(["image_id"])
)

silver_img.write.mode("overwrite").saveAsTable(f"{C}.silver.inspection_images")
print("✅ silver.inspection_images:", silver_img.count())

In [0]:
#checks
si = spark.table(f"{C}.silver.inspection_images")
qi = spark.table(f"{C}.silver.quality_inspections")

print("rows:", si.count(), " distinct image ids:", si.select("image_id").distinct().count())
print("confidence outside 0..1:", si.where((F.col("confidence") < 0) | (F.col("confidence") > 1)).count())
print("nulls in capture_ts or megapixels:",
      si.where(F.col("capture_ts").isNull() | F.col("megapixels").isNull()).count())

# Do the images agree with the inspections they belong to?
j = si.join(qi.select("inspection_id", F.col("part_number").alias("qi_part"), "defects_found"),
            "inspection_id", "left")
print("images with an unknown inspection:", j.where(F.col("qi_part").isNull()).count())
print("part number mismatches:", j.where(F.col("part_number") != F.col("qi_part")).count())
print("defect flag disagrees with inspection:",
      j.where(F.col("is_defect") != (F.col("defects_found") > 0)).count())

si.select("image_id", "image_type", "image_format", "megapixels", "is_defect",
          "confidence", "confidence_band").orderBy("image_id").show(truncate=False)